#### Transformer时序特征提取与XGBoost多模态预测框架

In [ ]:
import torch
import numpy as np 
import pandas as pd
import xgboost as xgb
from sklearn.model_selection  import TimeSeriesSplit 
from torch import nn
from sklearn.preprocessing  import RobustScaler 
from sklearn.metrics  import mean_absolute_percentage_error 
 
# 配置参数
CONFIG = {
    'seq_len': 60,          # 时间序列窗口 
    'transformer': {
        'd_model': 128,       
        'nhead': 8,
        'num_layers': 4,
        'dropout': 0.2
    },
    'xgb_params': {
        'max_depth': 7,
        'learning_rate': 0.02,
        'n_estimators': 1500,
        'subsample': 0.8
    }
} 

In [ ]:
class HybridModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.transformer  = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=CONFIG['transformer']['d_model'],
                nhead=CONFIG['transformer']['nhead'],
                dropout=CONFIG['transformer']['dropout']
            ), 
            num_layers=CONFIG['transformer']['num_layers']
        )
        self.feature_proj  = nn.Linear(8, CONFIG['transformer']['d_model'])
        
    def forward(self, ts_input):
        # 时序特征提取 
        ts_features = self.feature_proj(ts_input) 
        ts_features = ts_features.permute(1,  0, 2)
        transformer_out = self.transformer(ts_features) 
        return transformer_out[-1].squeeze()

In [ ]:
class MarketDataset(torch.utils.data.Dataset): 
    def __init__(self, stock_path, macro_path):
        # 加载股票数据与宏观数据
        stock_df = self._process_stock(stock_path)
        macro_df = self._process_macro(macro_path)
        
        # 对齐时间轴 
        self.full_data  = pd.merge_asof(stock_df.sort_index(),  
                                     macro_df.sort_index(), 
                                     left_index=True, 
                                     right_index=True)
        
        # 特征工程 
        self.scaler  = RobustScaler()
        self.ts_features  = self.scaler.fit_transform( 
            self.full_data[['open',  'high', 'low', 'close', 'volume', 
                           'macd', 'rsi', 'atr']].values)
        
        # 外部特征
        self.external_features  = self.full_data[['cpi',  'interest_rate',
                                                'industry_index', 
                                                'sentiment_score']].values
        
        self.targets  = self.full_data['close'].pct_change(24).shift(-24).values[24:-24] 
        
    def __getitem__(self, idx):
        ts_window = self.ts_features[idx:idx+CONFIG['seq_len']] 
        external = self.external_features[idx+CONFIG['seq_len']] 
        target = self.targets[idx+CONFIG['seq_len']] 
        return (torch.FloatTensor(ts_window),
                torch.FloatTensor(external),
                torch.FloatTensor([target]))
    
    def _add_technical_features(self, df):
        # 添加MACD、RSI、ATR等技术指标
        ...
        return df 
    
    def _process_macro(self, path):
        # 宏观经济数据处理 
        ...
        return df 

In [ ]:
# 两阶段训练流程
def train_pipeline():
    # 第一阶段：训练Transformer特征提取器 
    trans_model = HybridModel()
    trans_optim = torch.optim.AdamW(trans_model.parameters(),  lr=1e-4)
    
    # 时序特征预训练
    for epoch in range(10):
        for ts, _, target in train_loader:
            pred = trans_model(ts)
            loss = nn.MSELoss()(pred, target)
            loss.backward() 
            trans_optim.step() 
    
    # 第二阶段：XGBoost集成训练
    trans_model.eval() 
    train_features = []
    
    with torch.no_grad(): 
        for ts, ext, _ in train_loader:
            ts_feat = trans_model(ts).numpy()
            combined_feat = np.concatenate([ts_feat,  ext.numpy()],  axis=1)
            train_features.append(combined_feat) 
            
    X_train = np.vstack(train_features) 
    y_train = dataset.targets[CONFIG['seq_len']:len(train_features)+CONFIG['seq_len']] 
    
    xgb_model = xgb.XGBRegressor(**CONFIG['xgb_params'])
    xgb_model.fit(X_train,  y_train, 
                 eval_set=[(X_val, y_val)],
                 early_stopping_rounds=50)
    
    return trans_model, xgb_model

In [ ]:
# 预测模块 
def predict_next_day(model_pair, latest_data):
    trans_model, xgb_model = model_pair 
    ts_tensor = torch.FloatTensor(latest_data['ts_window'])
    ext_tensor = torch.FloatTensor(latest_data['external'])
    
    with torch.no_grad(): 
        ts_feat = trans_model(ts_tensor.unsqueeze(0)).numpy() 
        
    combined_feat = np.concatenate([ts_feat,  ext_tensor.numpy().reshape(1,-1)],  axis=1)
    return xgb_model.predict(combined_feat)[0] 

进完善

In [ ]:
def create_data_loaders(stock_path, macro_path, batch_size=64):
    # 完整数据集初始化 
    full_dataset = MarketDataset(stock_path, macro_path)
    
    # 时间序列交叉验证分割 
    tscv = TimeSeriesSplit(n_splits=5)
    train_index, val_index = list(tscv.split(full_dataset))[-1]   # 取最后一次分割 
    
    # 内存映射优化 
    train_sampler = torch.utils.data.SubsetRandomSampler(train_index) 
    val_sampler = torch.utils.data.SubsetRandomSampler(val_index) 
    
    # 多进程数据加载配置 
    num_workers = min(4, os.cpu_count()-1) 
    
    # 训练集加载器（关闭shuffle保持时序）
    train_loader = torch.utils.data.DataLoader( 
        full_dataset,
        batch_size=batch_size,
        sampler=train_sampler,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=True 
    )
    
    # 验证集加载器 
    val_loader = torch.utils.data.DataLoader( 
        full_dataset,
        batch_size=batch_size*2,  # 验证时可增大batch 
        sampler=val_sampler,
        num_workers=num_workers,
        pin_memory=True 
    )
    
    return train_loader, val_loader 

##### 内存优化策略

In [ ]:
class MemoryMappedDataset(MarketDataset):
    def __init__(self, *args):
        super().__init__(*args)
        # 将数据转为内存映射格式 
        self.ts_features  = np.memmap('ts_cache.npy',  dtype='float32', 
                                    mode='w+', shape=self.ts_features.shape) 
        self.external_features  = np.memmap('ext_cache.npy',  dtype='float32',
                                         mode='w+', shape=self.external_features.shape) 

##### 动态批处理技术

In [ ]:
def collate_fn(batch):
    ts_batch = torch.stack([item[0]  for item in batch])
    ext_batch = torch.stack([item[1]  for item in batch])
    target_batch = torch.cat([item[2]  for item in batch])
    
    # 时序维度对齐 
    if ts_batch.size(1)  < CONFIG['seq_len']:
        ts_batch = nn.functional.pad(ts_batch,  
                                   (0,0,0,CONFIG['seq_len']-ts_batch.size(1))) 
    
    return ts_batch, ext_batch, target_batch 

##### 生产环境集成方案

In [ ]:
if __name__ == "__main__":
    # 初始化数据管道 
    train_loader, val_loader = create_data_loaders(
        stock_path="2025_stock_data.parquet", 
        macro_path="macro_indicators.csv", 
        batch_size=128 
    )
    
    # 训练流程 
    trained_transformer, xgb_model = train_pipeline()
    
    # 实时预测示例 
    latest_data = {
        'ts_window': get_last_60_days_data(),  # 形状[60,8]
        'external': [2.8, 3.5, 115.2, 0.68]    # 当前宏观指标 
    }
    prediction = predict_next_day((trained_transformer, xgb_model), latest_data)
    print(f"2025年5月23日预测价格波动率: {prediction*100:.2f}%")

#####　建议部署时启用混合精度训练：

In [ ]:
scaler = torch.cuda.amp.GradScaler() 
with torch.cuda.amp.autocast(): 
    output = model(input)
    loss = criterion(output, target)
scaler.scale(loss).backward() 
scaler.step(optimizer) 

##### 部署建议

##### 1.实时数据管道架构

In [ ]:
class DataPipeline:
    def __init__(self):
        self.stream_processor  = ApacheKafkaConsumer()
        self.feature_store  = RedisFeatureDB()
        
    def update_features(self):
        """实时更新特征库"""
        while True:
            raw_data = self.stream_processor.get_new_data() 
            processed = self._process(raw_data)
            self.feature_store.update(processed) 
            
    def get_prediction_input(self):
        """生成预测所需输入格式"""
        ts_window = self.feature_store.get_last_n(CONFIG['seq_len']) 
        external = self.feature_store.get_current(['cpi','sentiment']) 
        return {
            'ts_window': scaler.transform(ts_window), 
            'external': external.values  
        }